In [7]:

# HEART DISEASE PREDICTION USING MLP


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report
)
import joblib
import warnings
warnings.filterwarnings("ignore")


# 1. LOAD AND PREPROCESS DATASET

df = pd.read_csv("processed_cardio_dataset.csv", sep=None, engine='python')

# Drop 'id' if present
if 'id' in df.columns:
    df = df.drop('id', axis=1)

# Convert True/False → 1/0
df = df.replace({True: 1, False: 0})

# Define features and label
label_col = 'cardio'
X = df.drop(label_col, axis=1)
y = df[label_col]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")
print("Features:", X.columns.tolist())


# 2. TRAIN BASELINE MLP MODEL


mlp_base = MLPClassifier(max_iter=300, early_stopping=True, random_state=42)
mlp_base.fit(X_train_s, y_train)
yb = mlp_base.predict(X_test_s)

print("\n=== Baseline MLP Metrics ===")
print("Accuracy :", round(accuracy_score(y_test, yb), 4))
print("F1 Score :", round(f1_score(y_test, yb), 4))


# 3. HYPERPARAMETER TUNING USING GRID SEARCH


param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(
    MLPClassifier(max_iter=500, early_stopping=True, random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train_s, y_train)

best_mlp = grid.best_estimator_
print("\n✅ Best MLP Parameters Found:", grid.best_params_)


# 4. EVALUATE BEST MODEL

yp = best_mlp.predict(X_test_s)
ypr = best_mlp.predict_proba(X_test_s)[:, 1]

print("\n=== Best MLP Model Metrics ===")
print("Accuracy :", round(accuracy_score(y_test, yp), 4))
print("Precision:", round(precision_score(y_test, yp), 4))
print("Recall   :", round(recall_score(y_test, yp), 4))
print("F1 Score :", round(f1_score(y_test, yp), 4))
print("AUC      :", round(roc_auc_score(y_test, ypr), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, yp))
print("\nClassification Report:\n", classification_report(y_test, yp))


# 5. SAVE TRAINED MODEL AND SCALER


joblib.dump(best_mlp, "mlp_heart_model.pkl")
joblib.dump(scaler, "mlp_scaler.pkl")
print("\n💾 Model and scaler saved successfully!")

# 6. TEST THE MODEL WITH EXAMPLE INPUT


# Automatically generate an example input with same number of features
example_input = np.zeros((1, X.shape[1]))  # all zeros
# Replace some example values for known columns if needed
# For example, if  dataset has ['age','gender','height','weight','ap_hi','ap_lo', ...]:
if 'age' in X.columns: example_input[0, X.columns.get_loc('age')] = 55
if 'gender' in X.columns: example_input[0, X.columns.get_loc('gender')] = 1
if 'height' in X.columns: example_input[0, X.columns.get_loc('height')] = 170
if 'weight' in X.columns: example_input[0, X.columns.get_loc('weight')] = 80
if 'ap_hi' in X.columns: example_input[0, X.columns.get_loc('ap_hi')] = 140
if 'ap_lo' in X.columns: example_input[0, X.columns.get_loc('ap_lo')] = 90
if 'cholesterol' in X.columns: example_input[0, X.columns.get_loc('cholesterol')] = 2
if 'gluc' in X.columns: example_input[0, X.columns.get_loc('gluc')] = 1
if 'smoke' in X.columns: example_input[0, X.columns.get_loc('smoke')] = 0
if 'alco' in X.columns: example_input[0, X.columns.get_loc('alco')] = 0
if 'active' in X.columns: example_input[0, X.columns.get_loc('active')] = 1

example_scaled = scaler.transform(example_input)
pred = best_mlp.predict(example_scaled)[0]
prob = best_mlp.predict_proba(example_scaled)[0][1]

print("\n=== Example Prediction ===")
if pred == 1:
    print(f"⚠️ Likely to have heart disease (Probability: {prob:.2f})")
else:
    print(f"✅ Unlikely to have heart disease (Probability: {prob:.2f})")


# 7. INTERACTIVE INPUT
print("\nWould you like to test manually? (y/n)")
choice = input().lower()

if choice == 'y':
    print("\nEnter patient details below:")
    new_data = np.zeros((1, X.shape[1]))
    for col in X.columns:
        val = input(f"{col}: ")
        if val.strip() == "":
            val = 0
        new_data[0, X.columns.get_loc(col)] = float(val)
    
    new_scaled = scaler.transform(new_data)
    pred = best_mlp.predict(new_scaled)[0]
    prob = best_mlp.predict_proba(new_scaled)[0][1]

    print()
    if pred == 1:
        print(f"⚠️ The person is likely to have heart disease. (Probability: {prob:.2f})")
    else:
        print(f"✅ The person is unlikely to have heart disease. (Probability: {prob:.2f})")

print("\n🎯 Done! Model trained, saved, and tested successfully.")


Training samples: 54999, Testing samples: 13750
Features: ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'smoke', 'alco', 'active', 'age_years', 'BMI', 'gender_Male', 'cholesterol_Normal', 'cholesterol_Well Above Normal', 'gluc_Normal', 'gluc_Well Above Normal']

=== Baseline MLP Metrics ===
Accuracy : 0.7327
F1 Score : 0.7267

✅ Best MLP Parameters Found: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (50,)}

=== Best MLP Model Metrics ===
Accuracy : 0.7301
Precision: 0.7429
Recall   : 0.6954
F1 Score : 0.7183
AUC      : 0.7941

Confusion Matrix:
 [[5307 1638]
 [2073 4732]]

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.76      0.74      6945
           1       0.74      0.70      0.72      6805

    accuracy                           0.73     13750
   macro avg       0.73      0.73      0.73     13750
weighted avg       0.73      0.73      0.73     13750


💾 Model and scaler saved successfully!

=== Example 

 y



Enter patient details below:


age:  12344
height:  176
weight:  57
ap_hi:  111
ap_lo:  90
smoke:  0
alco:  1
active:  1
age_years:  57
BMI:  43
gender_Male:  0
cholesterol_Normal:  0
cholesterol_Well Above Normal:  1
gluc_Normal:  0
gluc_Well Above Normal:  1



⚠️ The person is likely to have heart disease. (Probability: 0.56)

🎯 Done! Model trained, saved, and tested successfully.
